# SRP05 — Distribution-shift post-processing and verification

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Purpose.** Read the frozen SRP04b outputs and produce the compact task-level, shift-family, certification-cost, and validity tables used by the manuscript.

**Provenance.** Publication copy of the original executed SRP05 post-processing notebook. It does not retrain reservoirs. The scientific calculations are preserved; only the title and repository paths are cleaned for release.

**Execution note.** Embedded outputs are retained from the original full execution. Any absolute paths printed inside those historical outputs refer to the original workstation. Re-running this publication copy writes to the repository `results/reproduced/` directory.

In [1]:

# ============================================================
# Configuration and imports
# ============================================================
from __future__ import annotations

import os
import json
import math
import zipfile
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_SEED = 20260723
N_BOOT = int(os.environ.get("SRP05_N_BOOT", "30000"))
CI_LEVEL = 0.95
NUMERIC_TOL = 1e-9

# Optional explicit overrides. Leave as None for automatic discovery.
INPUT_DIR_OVERRIDE: Optional[str] = str((Path("../results/reproduced/SRP04b") if Path.cwd().name == "notebooks" else Path("results/reproduced/SRP04b")))
TASK_SUMMARY_FILE_OVERRIDE: Optional[str] = None
SAFETY_SUMMARY_FILE_OVERRIDE: Optional[str] = None
EPISODE_POLICY_FILE_OVERRIDE: Optional[str] = None

# Summary-only mode still exports exact task tables when no episode-level
# policy file can be identified. Cluster intervals and shift-family results
# are then marked unavailable rather than invented.
ALLOW_SUMMARY_ONLY = True

OUTPUT_DIR = Path(os.environ.get("SRP05_OUTPUT_DIR", str(Path("../results/reproduced/SRP05") if Path.cwd().name == "notebooks" else Path("results/reproduced/SRP05"))))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("SRP05 — manuscript integration audit")
print("bootstrap replicates:", N_BOOT)
print("output directory:", OUTPUT_DIR.resolve())


SRP05 — manuscript integration audit
bootstrap replicates: 30000
output directory: /home/ildefons/agentecon/SRP05_outputs


In [2]:

# ============================================================
# Locate SRP04b outputs and inventory every CSV
# ============================================================
def candidate_input_dirs() -> List[Path]:
    candidates: List[Path] = []

    if INPUT_DIR_OVERRIDE:
        candidates.append(Path(INPUT_DIR_OVERRIDE).expanduser())

    env_path = os.environ.get("SRP04B_OUTPUT_DIR")
    if env_path:
        candidates.append(Path(env_path).expanduser())

    here = Path.cwd()
    candidates.extend([
        here / "SRP04b_outputs",
        here.parent / "SRP04b_outputs",
        Path("/home/ildefons/agentecon/SRP04b_outputs"),
        Path("/mnt/data/SRP04b_outputs"),
    ])

    # Search a small number of parent levels without crawling the filesystem.
    for parent in [here, *list(here.parents)[:4]]:
        candidates.append(parent / "SRP04b_outputs")

    unique: List[Path] = []
    seen = set()
    for path in candidates:
        resolved = path.resolve()
        if str(resolved) not in seen:
            seen.add(str(resolved))
            unique.append(resolved)
    return unique


def discover_input_dir() -> Path:
    attempted = []
    for path in candidate_input_dirs():
        attempted.append(str(path))
        if path.is_dir() and any(path.rglob("*.csv")):
            return path

    message = (
        "Could not locate an SRP04b_outputs directory containing CSV files.\n"
        "Set INPUT_DIR_OVERRIDE in the configuration cell or define\n"
        "SRP04B_OUTPUT_DIR=/absolute/path/to/SRP04b_outputs.\n\n"
        "Attempted:\n- " + "\n- ".join(attempted)
    )
    raise FileNotFoundError(message)


INPUT_DIR = discover_input_dir()
print("input directory:", INPUT_DIR)

csv_paths = sorted(INPUT_DIR.rglob("*.csv"))
if not csv_paths:
    raise FileNotFoundError(f"No CSV files found below {INPUT_DIR}")

frames_raw: Dict[str, pd.DataFrame] = {}
inventory_rows = []

for path in csv_paths:
    relative = str(path.relative_to(INPUT_DIR))
    try:
        frame = pd.read_csv(path)
        frames_raw[relative] = frame
        inventory_rows.append({
            "relative_path": relative,
            "rows": int(len(frame)),
            "columns": int(len(frame.columns)),
            "column_names": " | ".join(map(str, frame.columns)),
            "read_status": "ok",
        })
    except Exception as exc:
        inventory_rows.append({
            "relative_path": relative,
            "rows": np.nan,
            "columns": np.nan,
            "column_names": "",
            "read_status": f"ERROR: {type(exc).__name__}: {exc}",
        })

inventory = pd.DataFrame(inventory_rows)
inventory.to_csv(OUTPUT_DIR / "srp05_input_inventory.csv", index=False)

print(f"read {len(frames_raw)} CSV files")
display(inventory)


input directory: /home/ildefons/agentecon/SRP04b_outputs
read 13 CSV files


,relative_path,rows,columns,column_names,read_status
0,srp04b_candidate_window_features.csv,54450,31,candidate | coord_logtau | coord_alpha | clean...,ok
1,srp04b_candidate_window_features_raw.csv,54450,30,candidate | coord_logtau | coord_alpha | clean...,ok
2,srp04b_configurations.csv,1089,8,task | seed | candidate | clean_val_nrmse | le...,ok
3,srp04b_configurations_raw.csv,1089,7,task | seed | candidate | clean_val_nrmse | le...,ok
4,srp04b_decision_gates.csv,10,2,gate | passed,ok
5,srp04b_family_domain_summary.csv,6,10,family | domain | historical_nrmse | oracle_sp...,ok
6,srp04b_meta_diagnostics.csv,18,6,task | seed | domain | candidate_count | gamma...,ok
7,srp04b_policy_results.csv,4680,9,task | seed | domain | window_id | family | se...,ok
8,srp04b_safe_regions.csv,1089,9,task | seed | candidate | clean_val_nrmse | hi...,ok
9,srp04b_safety_summary.csv,3,5,task | global_oracle_nrmse | safe_oracle_nrmse...,ok



## Schema normalization

SRP04b may have saved policy results in either wide form

| task | seed | family | domain | historical_nrmse | robust_sparse_nrmse | ... |

or long form

| task | seed | family | domain | policy | nrmse |

The next cell recognizes common aliases, creates a normalized copy, and never modifies the original CSVs.


In [3]:

# ============================================================
# Canonical column aliases and long-to-wide support
# ============================================================
COLUMN_ALIASES: Dict[str, Sequence[str]] = {
    "task": ("task", "dataset", "benchmark"),
    "seed": ("seed", "reservoir_seed", "random_seed"),
    "split": ("split", "data_split"),
    "family": ("family", "shift_family", "condition_family", "corruption_family"),
    "severity": ("severity", "shift_severity", "condition_severity"),
    "episode": ("episode", "episode_id", "window_index"),
    "window_id": ("window_id", "window", "episode_window_id"),
    "domain": ("domain", "action_space", "candidate_domain"),
    "historical_nrmse": (
        "historical_nrmse",
        "historical_best_nrmse",
        "historical_best",
        "historical",
    ),
    "robust_sparse_nrmse": (
        "robust_sparse_nrmse",
        "robust_sparse",
        "historically_fitted_sparse_nrmse",
        "historical_sparse_nrmse",
    ),
    "oracle_sparse_nrmse": (
        "oracle_sparse_nrmse",
        "oracle_sparse",
        "sparse_oracle_nrmse",
        "target_sparse_oracle_nrmse",
    ),
    "meta_top3_nrmse": (
        "meta_top3_nrmse",
        "meta_top3",
        "meta_risk_nrmse",
        "meta_selector_nrmse",
        "label_free_selector_nrmse",
    ),
    "relative_price_of_safety": (
        "relative_price_of_safety",
        "relative_price_of_certification",
        "relative_safety_cost",
        "relative_certification_cost",
    ),
}

POLICY_ALIASES: Dict[str, Sequence[str]] = {
    "historical_nrmse": (
        "historical_best",
        "historical",
        "historical_fixed",
        "fixed_historical",
    ),
    "robust_sparse_nrmse": (
        "robust_sparse",
        "historically_fitted_sparse",
        "historical_sparse",
    ),
    "oracle_sparse_nrmse": (
        "oracle_sparse",
        "sparse_oracle",
        "target_sparse_oracle",
    ),
    "meta_top3_nrmse": (
        "meta_top3",
        "meta_risk",
        "meta_selector",
        "label_free_selector",
    ),
}


def canonicalize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    lower_to_original = {str(col).strip().lower(): col for col in result.columns}
    rename = {}

    for canonical, aliases in COLUMN_ALIASES.items():
        if canonical in result.columns:
            continue
        matches = [
            lower_to_original[a.lower()]
            for a in aliases
            if a.lower() in lower_to_original
        ]
        if len(matches) == 1:
            rename[matches[0]] = canonical

    result = result.rename(columns=rename)

    if "domain" in result.columns:
        result["domain"] = (
            result["domain"]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({
                "safe": "certified",
                "safe_region": "certified",
                "hq": "certified",
                "historically_certified": "certified",
                "cert": "certified",
                "all": "global",
                "full": "global",
            })
        )

    return result


def maybe_pivot_policy_long(frame: pd.DataFrame) -> Optional[pd.DataFrame]:
    cols_lower = {str(c).strip().lower(): c for c in frame.columns}
    policy_col = next(
        (cols_lower[name] for name in ("policy", "method", "selector") if name in cols_lower),
        None,
    )
    value_col = next(
        (
            cols_lower[name]
            for name in ("nrmse", "loss", "heldout_nrmse", "test_nrmse")
            if name in cols_lower
        ),
        None,
    )

    if policy_col is None or value_col is None:
        return None

    base = canonicalize_columns(frame)
    policy_col = policy_col if policy_col in base.columns else str(policy_col)
    value_col = value_col if value_col in base.columns else str(value_col)

    id_candidates = [
        "task", "seed", "split", "family", "severity",
        "episode", "window_id", "domain",
    ]
    index_cols = [c for c in id_candidates if c in base.columns]
    if "task" not in index_cols or not index_cols:
        return None

    work = base[index_cols + [policy_col, value_col]].copy()
    work[policy_col] = work[policy_col].astype(str).str.strip().str.lower()

    policy_map = {}
    for canonical, aliases in POLICY_ALIASES.items():
        for alias in aliases:
            policy_map[alias.lower()] = canonical
    work["_canonical_policy"] = work[policy_col].map(policy_map)
    work = work.dropna(subset=["_canonical_policy"])
    if work.empty:
        return None

    pivot = (
        work.pivot_table(
            index=index_cols,
            columns="_canonical_policy",
            values=value_col,
            aggfunc="mean",
        )
        .reset_index()
    )
    pivot.columns.name = None
    return pivot


frames: Dict[str, pd.DataFrame] = {}
for name, raw in frames_raw.items():
    frames[name] = canonicalize_columns(raw)
    pivoted = maybe_pivot_policy_long(raw)
    if pivoted is not None:
        frames[name + "::PIVOTED"] = canonicalize_columns(pivoted)

print("normalized candidate tables:", len(frames))


normalized candidate tables: 14


In [4]:

# ============================================================
# Select source tables automatically, with explicit overrides
# ============================================================
TASK_REQUIRED = {
    "task", "domain", "historical_nrmse",
    "robust_sparse_nrmse", "oracle_sparse_nrmse",
}
EPISODE_REQUIRED = {
    "task", "seed", "family", "domain",
    "historical_nrmse", "robust_sparse_nrmse",
    "oracle_sparse_nrmse",
}
SAFETY_REQUIRED = {"task", "relative_price_of_safety"}


def resolve_override(name: Optional[str]) -> Optional[Tuple[str, pd.DataFrame]]:
    if name is None:
        return None

    options = [name, str(Path(name).as_posix())]
    for option in options:
        if option in frames:
            return option, frames[option].copy()
    raise KeyError(
        f"Override {name!r} was not found. Available normalized tables:\n"
        + "\n".join(sorted(frames))
    )


def rank_table(
    required: set,
    preferred: Sequence[str],
    exclude_min_rows: Optional[int] = None,
) -> List[Tuple[float, str, pd.DataFrame]]:
    ranked = []
    for name, frame in frames.items():
        columns = set(frame.columns)
        if not required.issubset(columns):
            continue
        if exclude_min_rows is not None and len(frame) < exclude_min_rows:
            continue

        score = 100.0 * len(required)
        score += 5.0 * sum(col in columns for col in preferred)
        score += min(math.log10(max(len(frame), 1)), 6.0)
        if "::PIVOTED" in name:
            score += 0.5
        ranked.append((score, name, frame.copy()))

    return sorted(ranked, key=lambda item: (-item[0], item[1]))


task_override = resolve_override(TASK_SUMMARY_FILE_OVERRIDE)
episode_override = resolve_override(EPISODE_POLICY_FILE_OVERRIDE)
safety_override = resolve_override(SAFETY_SUMMARY_FILE_OVERRIDE)

if task_override is None:
    task_candidates = rank_table(
        TASK_REQUIRED,
        preferred=("meta_top3_nrmse", "relative_price_of_safety"),
    )
    # Prefer compact task/domain summaries over episode-level files.
    task_candidates = sorted(
        task_candidates,
        key=lambda item: (
            item[2]["seed"].notna().any() if "seed" in item[2].columns else False,
            len(item[2]),
            -item[0],
            item[1],
        ),
    )
    task_source = task_candidates[0] if task_candidates else None
else:
    task_source = (np.inf, task_override[0], task_override[1])

if episode_override is None:
    episode_candidates = rank_table(
        EPISODE_REQUIRED,
        preferred=("severity", "episode", "window_id", "split", "meta_top3_nrmse"),
    )
    episode_source = episode_candidates[0] if episode_candidates else None
else:
    episode_source = (np.inf, episode_override[0], episode_override[1])

if safety_override is None:
    safety_candidates = rank_table(
        SAFETY_REQUIRED,
        preferred=("global_oracle_nrmse", "certified_oracle_nrmse"),
    )
    safety_source = safety_candidates[0] if safety_candidates else None
else:
    safety_source = (np.inf, safety_override[0], safety_override[1])

if task_source is None:
    raise RuntimeError(
        "No task/domain summary table could be identified. "
        f"Required canonical columns: {sorted(TASK_REQUIRED)}"
    )

task_source_name = task_source[1]
task_summary_raw = task_source[2].copy()

episode_source_name = episode_source[1] if episode_source else None
episode_policy_raw = episode_source[2].copy() if episode_source else None

safety_source_name = safety_source[1] if safety_source else None
safety_summary_raw = safety_source[2].copy() if safety_source else None

print("task summary source:", task_source_name)
print("episode policy source:", episode_source_name or "NOT FOUND")
print("safety summary source:", safety_source_name or "NOT FOUND")

display(task_summary_raw.head())
if episode_policy_raw is not None:
    display(episode_policy_raw.head())
if safety_summary_raw is not None:
    display(safety_summary_raw.head())

if episode_policy_raw is None and not ALLOW_SUMMARY_ONLY:
    raise RuntimeError(
        "No episode-level policy table found and ALLOW_SUMMARY_ONLY=False."
    )


task summary source: srp04b_task_domain_summary.csv
episode policy source: srp04b_policy_results.csv::PIVOTED
safety summary source: srp04b_safety_summary.csv


,task,domain,historical_nrmse,oracle_sparse_nrmse,robust_sparse_nrmse,meta_top3_nrmse,oracle_relative_gain,robust_relative_gain,meta_relative_gain,robust_capture_fraction,meta_capture_fraction,robust_harm_fraction,meta_harm_fraction,oracle_valid
0,lorenz_x,global,0.657726,0.436730,0.527367,0.542231,0.334285,0.195919,0.175158,-2.522643,-0.874673,0.450000,0.433333,True
1,lorenz_x,certified,0.718076,0.458480,0.527275,0.556062,0.359535,0.261929,0.225998,-5.697078,-2.868432,0.383333,0.500000,True
2,mackey_glass,global,0.619260,0.377551,0.495867,0.540338,0.384976,0.196535,0.122175,-1.997236,-1.612620,0.366667,0.500000,True
3,mackey_glass,certified,0.526804,0.413852,0.501914,0.524957,0.213230,0.045413,0.000717,-1.072191,-1.540635,0.466667,0.483333,True
4,narma10,global,1.006426,0.751775,0.885420,0.913541,0.250695,0.119227,0.091883,-0.011939,0.199244,0.200000,0.116667,True


,task,seed,family,severity,episode,window_id,domain,historical_nrmse,oracle_sparse_nrmse,robust_sparse_nrmse
0,lorenz_x,0,additive_ar1,1.1,0,lorenz_x_seed0_test_additive_ar1_s1.10_ep0,certified,1.980891,0.906326,0.942505
1,lorenz_x,0,additive_ar1,1.1,0,lorenz_x_seed0_test_additive_ar1_s1.10_ep0,global,1.850692,0.883491,0.948199
2,lorenz_x,0,additive_ar1,1.1,1,lorenz_x_seed0_test_additive_ar1_s1.10_ep1,certified,1.935751,0.834661,0.913590
3,lorenz_x,0,additive_ar1,1.1,1,lorenz_x_seed0_test_additive_ar1_s1.10_ep1,global,1.811311,0.808108,0.918116
4,lorenz_x,0,additive_ar1,1.1,2,lorenz_x_seed0_test_additive_ar1_s1.10_ep2,certified,1.966595,0.995993,1.054175


,task,global_oracle_nrmse,safe_oracle_nrmse,price_of_safety,relative_price_of_safety
0,lorenz_x,0.436730,0.458480,0.021750,0.069175
1,mackey_glass,0.377551,0.413852,0.036301,0.129902
2,narma10,0.751775,0.814078,0.062303,0.071500


In [5]:

# ============================================================
# Clean and validate selected tables
# ============================================================
KEY_LOSS_COLUMNS = [
    "historical_nrmse",
    "robust_sparse_nrmse",
    "oracle_sparse_nrmse",
]
OPTIONAL_LOSS_COLUMNS = ["meta_top3_nrmse"]


def numericize(frame: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:
    result = frame.copy()
    for column in columns:
        if column in result.columns:
            result[column] = pd.to_numeric(result[column], errors="coerce")
    return result


task_summary = numericize(
    task_summary_raw,
    KEY_LOSS_COLUMNS + OPTIONAL_LOSS_COLUMNS + ["relative_price_of_safety"],
)
task_summary["task"] = task_summary["task"].astype(str)
task_summary["domain"] = task_summary["domain"].astype(str).str.lower().replace({"safe": "certified"})

if "split" in task_summary.columns:
    test_mask = task_summary["split"].astype(str).str.lower().eq("test")
    if test_mask.any():
        task_summary = task_summary[test_mask].copy()

# Collapse accidental duplicates conservatively by arithmetic mean.
task_group_cols = ["task", "domain"]
task_numeric_cols = [
    c for c in KEY_LOSS_COLUMNS + OPTIONAL_LOSS_COLUMNS
    if c in task_summary.columns
]
task_summary = (
    task_summary.groupby(task_group_cols, as_index=False)[task_numeric_cols]
    .mean()
)

episode_policy = None
INPUT_DUPLICATE_RECORD_COUNT = 0
if episode_policy_raw is not None:
    episode_policy = numericize(
        episode_policy_raw,
        KEY_LOSS_COLUMNS + OPTIONAL_LOSS_COLUMNS + ["severity", "episode"],
    )
    episode_policy["task"] = episode_policy["task"].astype(str)
    episode_policy["family"] = episode_policy["family"].astype(str)
    episode_policy["domain"] = (
        episode_policy["domain"]
        .astype(str)
        .str.lower()
        .replace({"safe": "certified"})
    )

    if "split" in episode_policy.columns:
        test_mask = episode_policy["split"].astype(str).str.lower().eq("test")
        if test_mask.any():
            episode_policy = episode_policy[test_mask].copy()

    if "window_id" not in episode_policy.columns:
        parts = [
            episode_policy["task"].astype(str),
            episode_policy["seed"].astype(str),
            episode_policy["family"].astype(str),
        ]
        if "severity" in episode_policy.columns:
            parts.append(episode_policy["severity"].astype(str))
        if "episode" in episode_policy.columns:
            parts.append(episode_policy["episode"].astype(str))
        parts.append(episode_policy["domain"].astype(str))
        episode_policy["window_id"] = parts[0]
        for part in parts[1:]:
            episode_policy["window_id"] = episode_policy["window_id"] + "|" + part

    episode_policy["task_seed"] = (
        episode_policy["task"].astype(str)
        + "|seed="
        + episode_policy["seed"].astype(str)
    )

    # Record exact duplicate episode/domain records before deduplication.
    dedupe_cols = [
        c for c in [
            "task", "seed", "family", "severity",
            "episode", "window_id", "domain",
        ]
        if c in episode_policy.columns
    ]
    INPUT_DUPLICATE_RECORD_COUNT = int(
        episode_policy.duplicated(
            subset=dedupe_cols,
            keep=False,
        ).sum()
    )
    episode_policy = episode_policy.drop_duplicates(
        subset=dedupe_cols,
        keep="first",
    ).reset_index(drop=True)

print("normalized task summary")
display(task_summary)

if episode_policy is not None:
    print("episode-level rows:", len(episode_policy))
    print("tasks:", sorted(episode_policy["task"].unique()))
    print("seeds:", sorted(episode_policy["seed"].unique()))
    print("families:", sorted(episode_policy["family"].unique()))
    print("domains:", sorted(episode_policy["domain"].unique()))


normalized task summary


,task,domain,historical_nrmse,robust_sparse_nrmse,oracle_sparse_nrmse,meta_top3_nrmse
0,lorenz_x,certified,0.718076,0.527275,0.458480,0.556062
1,lorenz_x,global,0.657726,0.527367,0.436730,0.542231
2,mackey_glass,certified,0.526804,0.501914,0.413852,0.524957
3,mackey_glass,global,0.619260,0.495867,0.377551,0.540338
4,narma10,certified,1.006426,0.894911,0.814078,0.913535
5,narma10,global,1.006426,0.885420,0.751775,0.913541


episode-level rows: 360
tasks: ['lorenz_x', 'mackey_glass', 'narma10']
seeds: [np.int64(0), np.int64(1), np.int64(2)]
families: ['additive_ar1', 'block_missing', 'offset_drift']
domains: ['certified', 'global']


In [6]:

import hashlib

# ============================================================
# Statistics helpers: aggregate gains and clustered bootstrap
# ============================================================

def stable_seed(*parts: object) -> int:
    payload = "||".join(map(str, parts)).encode("utf-8")
    digest = hashlib.sha256(payload).digest()
    return RANDOM_SEED + int.from_bytes(digest[:4], "little") % 1000000


def relative_gain_from_means(
    historical: np.ndarray,
    policy: np.ndarray,
) -> float:
    historical = np.asarray(historical, dtype=float)
    policy = np.asarray(policy, dtype=float)
    mask = np.isfinite(historical) & np.isfinite(policy)
    if not np.any(mask):
        return np.nan
    h = float(np.mean(historical[mask]))
    p = float(np.mean(policy[mask]))
    return (h - p) / h if abs(h) > 1e-15 else np.nan


def certification_cost_from_means(
    global_oracle: np.ndarray,
    certified_oracle: np.ndarray,
) -> float:
    global_oracle = np.asarray(global_oracle, dtype=float)
    certified_oracle = np.asarray(certified_oracle, dtype=float)
    mask = np.isfinite(global_oracle) & np.isfinite(certified_oracle)
    if not np.any(mask):
        return np.nan
    g = float(np.mean(global_oracle[mask]))
    c = float(np.mean(certified_oracle[mask]))
    return (c - g) / g if abs(g) > 1e-15 else np.nan


def bootstrap_cluster_ratio(
    cluster_table: pd.DataFrame,
    numerator_column: str,
    denominator_column: str,
    statistic: Callable[[np.ndarray, np.ndarray], float],
    n_boot: int = N_BOOT,
    seed: int = RANDOM_SEED,
) -> Tuple[float, float, float, int]:
    work = cluster_table[
        [numerator_column, denominator_column]
    ].replace([np.inf, -np.inf], np.nan).dropna()

    n_clusters = len(work)
    if n_clusters == 0:
        return np.nan, np.nan, np.nan, 0

    a = work[numerator_column].to_numpy(float)
    b = work[denominator_column].to_numpy(float)
    point = statistic(a, b)

    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        indices = rng.integers(0, n_clusters, size=n_clusters)
        boot[i] = statistic(a[indices], b[indices])

    boot = boot[np.isfinite(boot)]
    if len(boot) == 0:
        return point, np.nan, np.nan, n_clusters

    alpha = 1.0 - CI_LEVEL
    lower, upper = np.quantile(
        boot,
        [alpha / 2.0, 1.0 - alpha / 2.0],
    )
    return point, float(lower), float(upper), n_clusters


def format_ci(value: float, lower: float, upper: float, digits: int = 3) -> str:
    if not all(np.isfinite([value, lower, upper])):
        return "unavailable"
    return f"{value:.{digits}f} [{lower:.{digits}f}, {upper:.{digits}f}]"


In [7]:

# ============================================================
# Task-level manuscript table
# ============================================================
global_summary = task_summary[
    task_summary["domain"].eq("global")
].copy()

if global_summary.empty:
    raise RuntimeError("The selected task summary contains no global-domain rows.")

task_rows = []
for row in global_summary.itertuples(index=False):
    historical = float(row.historical_nrmse)
    robust = float(row.robust_sparse_nrmse)
    oracle = float(row.oracle_sparse_nrmse)
    meta = float(row.meta_top3_nrmse) if hasattr(row, "meta_top3_nrmse") else np.nan

    task_rows.append({
        "task": row.task,
        "historical_best_fixed_nrmse": historical,
        "historically_fitted_sparse_nrmse": robust,
        "label_free_meta_selector_nrmse": meta,
        "target_fitted_sparse_oracle_nrmse": oracle,
        "historical_sparse_relative_gain": (
            (historical - robust) / historical
            if abs(historical) > 1e-15 else np.nan
        ),
        "meta_selector_relative_gain": (
            (historical - meta) / historical
            if np.isfinite(meta) and abs(historical) > 1e-15 else np.nan
        ),
        "target_oracle_relative_gain": (
            (historical - oracle) / historical
            if abs(historical) > 1e-15 else np.nan
        ),
        "oracle_gap_remaining_after_historical_sparse": (
            (robust - oracle) / historical
            if abs(historical) > 1e-15 else np.nan
        ),
    })

task_table = pd.DataFrame(task_rows)

# Add descriptive seed-cluster intervals when episode-level data exist.
for metric_name, policy_column in [
    ("historical_sparse_relative_gain", "robust_sparse_nrmse"),
    ("meta_selector_relative_gain", "meta_top3_nrmse"),
    ("target_oracle_relative_gain", "oracle_sparse_nrmse"),
]:
    task_table[metric_name + "_ci_low"] = np.nan
    task_table[metric_name + "_ci_high"] = np.nan
    task_table[metric_name + "_n_seed_clusters"] = np.nan

    if episode_policy is None or policy_column not in episode_policy.columns:
        continue

    for task in task_table["task"]:
        subset = episode_policy[
            episode_policy["domain"].eq("global")
            & episode_policy["task"].eq(task)
        ].copy()
        if subset.empty:
            continue

        seed_table = (
            subset.groupby("seed", as_index=False)[
                ["historical_nrmse", policy_column]
            ]
            .mean()
        )
        point, low, high, n_clusters = bootstrap_cluster_ratio(
            seed_table,
            "historical_nrmse",
            policy_column,
            relative_gain_from_means,
            seed=stable_seed("task", task, metric_name),
        )
        mask = task_table["task"].eq(task)
        # Preserve exact summary-derived point estimate in the main column.
        task_table.loc[mask, metric_name + "_ci_low"] = low
        task_table.loc[mask, metric_name + "_ci_high"] = high
        task_table.loc[mask, metric_name + "_n_seed_clusters"] = n_clusters

task_table.to_csv(OUTPUT_DIR / "srp05_task_table.csv", index=False)
print("Task-level manuscript table")
display(task_table)


Task-level manuscript table


,task,historical_best_fixed_nrmse,historically_fitted_sparse_nrmse,label_free_meta_selector_nrmse,target_fitted_sparse_oracle_nrmse,historical_sparse_relative_gain,meta_selector_relative_gain,target_oracle_relative_gain,oracle_gap_remaining_after_historical_sparse,historical_sparse_relative_gain_ci_low,historical_sparse_relative_gain_ci_high,historical_sparse_relative_gain_n_seed_clusters,meta_selector_relative_gain_ci_low,meta_selector_relative_gain_ci_high,meta_selector_relative_gain_n_seed_clusters,target_oracle_relative_gain_ci_low,target_oracle_relative_gain_ci_high,target_oracle_relative_gain_n_seed_clusters
0,lorenz_x,0.657726,0.527367,0.542231,0.436730,0.198197,0.175597,0.336000,0.137803,0.153329,0.234776,3.0,NaN,NaN,NaN,0.300965,0.397810,3.0
1,mackey_glass,0.619260,0.495867,0.540338,0.377551,0.199259,0.127446,0.390319,0.191060,0.158166,0.252654,3.0,NaN,NaN,NaN,0.292533,0.433960,3.0
2,narma10,1.006426,0.885420,0.913541,0.751775,0.120233,0.092292,0.253025,0.132792,0.092508,0.133500,3.0,NaN,NaN,NaN,0.190444,0.287064,3.0


In [8]:

# ============================================================
# Meta-risk selector table
# ============================================================
meta_columns = [
    "task",
    "historical_best_fixed_nrmse",
    "historically_fitted_sparse_nrmse",
    "label_free_meta_selector_nrmse",
    "meta_selector_relative_gain",
    "meta_selector_relative_gain_ci_low",
    "meta_selector_relative_gain_ci_high",
    "meta_selector_relative_gain_n_seed_clusters",
]
meta_columns = [c for c in meta_columns if c in task_table.columns]
meta_risk_table = task_table[meta_columns].copy()
meta_risk_table.to_csv(OUTPUT_DIR / "srp05_meta_risk_table.csv", index=False)

print("Label-free meta-risk selector table")
display(meta_risk_table)


Label-free meta-risk selector table


,task,historical_best_fixed_nrmse,historically_fitted_sparse_nrmse,label_free_meta_selector_nrmse,meta_selector_relative_gain,meta_selector_relative_gain_ci_low,meta_selector_relative_gain_ci_high,meta_selector_relative_gain_n_seed_clusters
0,lorenz_x,0.657726,0.527367,0.542231,0.175597,NaN,NaN,NaN
1,mackey_glass,0.619260,0.495867,0.540338,0.127446,NaN,NaN,NaN
2,narma10,1.006426,0.885420,0.913541,0.092292,NaN,NaN,NaN


In [9]:

# ============================================================
# Shift-family table with task–seed clustered uncertainty
# ============================================================
shift_family_table = pd.DataFrame()

if episode_policy is not None:
    global_episode = episode_policy[
        episode_policy["domain"].eq("global")
    ].copy()

    family_rows = []
    for family, subset in global_episode.groupby("family", sort=True):
        cluster_table = (
            subset.groupby("task_seed", as_index=False)[
                ["historical_nrmse", "robust_sparse_nrmse"]
            ]
            .mean()
        )

        point, low, high, n_clusters = bootstrap_cluster_ratio(
            cluster_table,
            "historical_nrmse",
            "robust_sparse_nrmse",
            relative_gain_from_means,
            seed=stable_seed("family", family),
        )

        episode_harm = (
            subset["robust_sparse_nrmse"]
            > subset["historical_nrmse"] + NUMERIC_TOL
        )
        cluster_harm = (
            cluster_table["robust_sparse_nrmse"]
            > cluster_table["historical_nrmse"] + NUMERIC_TOL
        )

        family_rows.append({
            "shift_family": family,
            "historical_best_fixed_nrmse": float(subset["historical_nrmse"].mean()),
            "historically_fitted_sparse_nrmse": float(subset["robust_sparse_nrmse"].mean()),
            "historical_sparse_relative_gain": point,
            "historical_sparse_relative_gain_ci_low": low,
            "historical_sparse_relative_gain_ci_high": high,
            "episode_harm_fraction": float(episode_harm.mean()),
            "episode_harm_count": int(episode_harm.sum()),
            "episode_window_count": int(len(subset)),
            "cluster_harm_fraction": float(cluster_harm.mean()),
            "cluster_harm_count": int(cluster_harm.sum()),
            "task_seed_cluster_count": int(n_clusters),
            "task_count": int(subset["task"].nunique()),
            "seed_count": int(subset["seed"].nunique()),
        })

    shift_family_table = pd.DataFrame(family_rows)
    shift_family_table.to_csv(
        OUTPUT_DIR / "srp05_shift_family_table.csv",
        index=False,
    )
    print("Shift-family manuscript table")
    display(shift_family_table)
else:
    pd.DataFrame([{
        "status": "unavailable",
        "reason": (
            "No episode-level policy table was identified. "
            "Task-level exact summaries were still exported."
        ),
    }]).to_csv(
        OUTPUT_DIR / "srp05_shift_family_table.csv",
        index=False,
    )
    print("Shift-family analysis unavailable: no episode-level policy table.")


Shift-family manuscript table


,shift_family,historical_best_fixed_nrmse,historically_fitted_sparse_nrmse,historical_sparse_relative_gain,historical_sparse_relative_gain_ci_low,historical_sparse_relative_gain_ci_high,episode_harm_fraction,episode_harm_count,episode_window_count,cluster_harm_fraction,cluster_harm_count,task_seed_cluster_count,task_count,seed_count
0,additive_ar1,1.460592,0.977092,0.331030,0.230798,0.416113,0.000000,0,36,0.000000,0,9,3,3
1,block_missing,0.477566,0.508831,-0.065466,-0.151088,-0.004883,0.625000,45,72,0.777778,7,9,3,3
2,offset_drift,0.694981,0.593168,0.146498,0.111800,0.168237,0.222222,16,72,0.000000,0,9,3,3


In [10]:

# ============================================================
# Price of certification: global versus certified oracle
# ============================================================
certification_rows = []

cert_summary = task_summary[
    task_summary["domain"].eq("certified")
].copy()

for task in sorted(set(global_summary["task"]) & set(cert_summary["task"])):
    global_row = global_summary[global_summary["task"].eq(task)].iloc[0]
    cert_row = cert_summary[cert_summary["task"].eq(task)].iloc[0]

    global_oracle = float(global_row["oracle_sparse_nrmse"])
    cert_oracle = float(cert_row["oracle_sparse_nrmse"])
    recomputed = (
        (cert_oracle - global_oracle) / global_oracle
        if abs(global_oracle) > 1e-15 else np.nan
    )

    source_value = np.nan
    if safety_summary_raw is not None:
        safety_work = canonicalize_columns(safety_summary_raw)
        safety_work = numericize(safety_work, ["relative_price_of_safety"])
        match = safety_work[safety_work["task"].astype(str).eq(str(task))]
        if not match.empty and "relative_price_of_safety" in match.columns:
            source_value = float(match.iloc[0]["relative_price_of_safety"])

    ci_low = ci_high = np.nan
    n_clusters = 0

    if episode_policy is not None:
        g = episode_policy[
            episode_policy["domain"].eq("global")
            & episode_policy["task"].eq(task)
        ][["window_id", "seed", "oracle_sparse_nrmse"]].rename(
            columns={"oracle_sparse_nrmse": "global_oracle"}
        )
        c = episode_policy[
            episode_policy["domain"].eq("certified")
            & episode_policy["task"].eq(task)
        ][["window_id", "seed", "oracle_sparse_nrmse"]].rename(
            columns={"oracle_sparse_nrmse": "certified_oracle"}
        )

        # window_id may encode the domain. Fall back to matched descriptive keys.
        merged = g.merge(c, on=["window_id", "seed"], how="inner")
        if merged.empty:
            keys = [
                key for key in ["task", "seed", "family", "severity", "episode"]
                if key in episode_policy.columns
            ]
            g2 = episode_policy[
                episode_policy["domain"].eq("global")
                & episode_policy["task"].eq(task)
            ][keys + ["oracle_sparse_nrmse"]].rename(
                columns={"oracle_sparse_nrmse": "global_oracle"}
            )
            c2 = episode_policy[
                episode_policy["domain"].eq("certified")
                & episode_policy["task"].eq(task)
            ][keys + ["oracle_sparse_nrmse"]].rename(
                columns={"oracle_sparse_nrmse": "certified_oracle"}
            )
            merged = g2.merge(c2, on=keys, how="inner")

        if not merged.empty:
            seed_table = (
                merged.groupby("seed", as_index=False)[
                    ["global_oracle", "certified_oracle"]
                ]
                .mean()
            )
            _, ci_low, ci_high, n_clusters = bootstrap_cluster_ratio(
                seed_table,
                "global_oracle",
                "certified_oracle",
                certification_cost_from_means,
                seed=stable_seed("certification", task),
            )

    certification_rows.append({
        "task": task,
        "global_target_sparse_oracle_nrmse": global_oracle,
        "certified_target_sparse_oracle_nrmse": cert_oracle,
        "relative_certification_cost_recomputed": recomputed,
        "relative_certification_cost_source": source_value,
        "source_minus_recomputed": (
            source_value - recomputed
            if np.isfinite(source_value) and np.isfinite(recomputed)
            else np.nan
        ),
        "relative_certification_cost_ci_low": ci_low,
        "relative_certification_cost_ci_high": ci_high,
        "n_seed_clusters": n_clusters,
    })

certification_table = pd.DataFrame(certification_rows)
certification_table.to_csv(
    OUTPUT_DIR / "srp05_certification_cost_table.csv",
    index=False,
)

print("Certification-cost table")
display(certification_table)


Certification-cost table


,task,global_target_sparse_oracle_nrmse,certified_target_sparse_oracle_nrmse,relative_certification_cost_recomputed,relative_certification_cost_source,source_minus_recomputed,relative_certification_cost_ci_low,relative_certification_cost_ci_high,n_seed_clusters
0,lorenz_x,0.436730,0.458480,0.049801,0.069175,0.019374,0.030239,0.089294,3
1,mackey_glass,0.377551,0.413852,0.096148,0.129902,0.033754,0.070510,0.143749,3
2,narma10,0.751775,0.814078,0.082875,0.071500,-0.011375,0.018640,0.167563,3


In [11]:

# ============================================================
# Per-task/seed metrics and validity checks
# ============================================================
task_seed_metrics = pd.DataFrame()
checks = []

def add_check(
    name: str,
    passed: Optional[bool],
    n_violations: Optional[int],
    n_tested: Optional[int],
    detail: str,
):
    checks.append({
        "check": name,
        "passed": passed,
        "n_violations": n_violations,
        "n_tested": n_tested,
        "detail": detail,
    })


# Summary-level finite-value checks
summary_key = task_summary[
    ["task", "domain"] + task_numeric_cols
].copy()
summary_nan_count = int(summary_key[task_numeric_cols].isna().sum().sum())
add_check(
    "No missing key values in task summary",
    summary_nan_count == 0,
    summary_nan_count,
    int(summary_key[task_numeric_cols].size),
    "Checks historical, historical-sparse, oracle, and optional meta losses.",
)

# Matched-domain oracle validity from summary
for domain in ["global", "certified"]:
    subset = task_summary[task_summary["domain"].eq(domain)]
    if subset.empty:
        add_check(
            f"{domain}: oracle no worse than historical fixed",
            None,
            None,
            0,
            "Domain not present in selected task summary.",
        )
        continue

    violations = (
        subset["oracle_sparse_nrmse"]
        > subset["historical_nrmse"] + NUMERIC_TOL
    )
    add_check(
        f"{domain}: oracle no worse than historical fixed",
        bool((~violations).all()),
        int(violations.sum()),
        int(len(subset)),
        "Matched action-space summary check.",
    )

if not certification_table.empty:
    violations = (
        certification_table["global_target_sparse_oracle_nrmse"]
        > certification_table["certified_target_sparse_oracle_nrmse"] + NUMERIC_TOL
    )
    add_check(
        "Global oracle no worse than certified oracle",
        bool((~violations).all()),
        int(violations.sum()),
        int(len(certification_table)),
        "The global candidate set contains the certified subset.",
    )

if episode_policy is not None:
    metric_columns = [
        "historical_nrmse",
        "robust_sparse_nrmse",
        "oracle_sparse_nrmse",
    ]
    if "meta_top3_nrmse" in episode_policy.columns:
        metric_columns.append("meta_top3_nrmse")

    task_seed_metrics = (
        episode_policy.groupby(
            ["task", "seed", "domain"],
            as_index=False,
        )[metric_columns]
        .mean()
    )

    task_seed_metrics["historical_sparse_relative_gain"] = (
        task_seed_metrics["historical_nrmse"]
        - task_seed_metrics["robust_sparse_nrmse"]
    ) / task_seed_metrics["historical_nrmse"]

    task_seed_metrics["target_oracle_relative_gain"] = (
        task_seed_metrics["historical_nrmse"]
        - task_seed_metrics["oracle_sparse_nrmse"]
    ) / task_seed_metrics["historical_nrmse"]

    if "meta_top3_nrmse" in task_seed_metrics.columns:
        task_seed_metrics["meta_selector_relative_gain"] = (
            task_seed_metrics["historical_nrmse"]
            - task_seed_metrics["meta_top3_nrmse"]
        ) / task_seed_metrics["historical_nrmse"]

    for domain in ["global", "certified"]:
        subset = episode_policy[episode_policy["domain"].eq(domain)]
        if subset.empty:
            add_check(
                f"{domain}: episode oracle no worse than historical fixed",
                None,
                None,
                0,
                "Domain not present in episode-level source.",
            )
            continue

        violations = (
            subset["oracle_sparse_nrmse"]
            > subset["historical_nrmse"] + NUMERIC_TOL
        )
        add_check(
            f"{domain}: episode oracle no worse than historical fixed",
            bool((~violations).all()),
            int(violations.sum()),
            int(len(subset)),
            "Matched action-space episode-level check.",
        )

    key_nan_count = int(episode_policy[metric_columns].isna().sum().sum())
    add_check(
        "No missing key values in episode policy table",
        key_nan_count == 0,
        key_nan_count,
        int(episode_policy[metric_columns].size),
        "Checks all policy NRMSE fields used by SRP05.",
    )

    duplicate_count = int(INPUT_DUPLICATE_RECORD_COUNT)
    add_check(
        "No duplicate episode/domain records",
        duplicate_count == 0,
        duplicate_count,
        int(len(episode_policy)),
        "Duplicate records can otherwise overweight particular windows.",
    )

    add_check(
        "Expected task count",
        episode_policy["task"].nunique() == 3,
        abs(int(episode_policy["task"].nunique()) - 3),
        3,
        f"Observed tasks: {sorted(episode_policy['task'].unique())}",
    )
    add_check(
        "Expected reservoir-seed count",
        episode_policy["seed"].nunique() == 3,
        abs(int(episode_policy["seed"].nunique()) - 3),
        3,
        f"Observed seeds: {sorted(episode_policy['seed'].unique())}",
    )
else:
    add_check(
        "Episode-level policy table available",
        False,
        1,
        1,
        "Summary-only mode: clustered intervals and shift-family analysis are unavailable.",
    )

task_seed_metrics.to_csv(
    OUTPUT_DIR / "srp05_task_seed_metrics.csv",
    index=False,
)

validity_checks = pd.DataFrame(checks)
validity_checks.to_csv(
    OUTPUT_DIR / "srp05_validity_checks.csv",
    index=False,
)

print("Validity checks")
display(validity_checks)

failed = validity_checks[validity_checks["passed"].eq(False)]
if not failed.empty:
    print(
        "\nWARNING: One or more checks failed or an expected source was unavailable. "
        "Read srp05_validity_checks.csv before using manuscript statements."
    )


Validity checks


,check,passed,n_violations,n_tested,detail
0,No missing key values in task summary,True,0,24,"Checks historical, historical-sparse, oracle, ..."
1,global: oracle no worse than historical fixed,True,0,3,Matched action-space summary check.
2,certified: oracle no worse than historical fixed,True,0,3,Matched action-space summary check.
3,Global oracle no worse than certified oracle,True,0,3,The global candidate set contains the certifie...
4,global: episode oracle no worse than historica...,True,0,180,Matched action-space episode-level check.
5,certified: episode oracle no worse than histor...,True,0,180,Matched action-space episode-level check.
6,No missing key values in episode policy table,True,0,1080,Checks all policy NRMSE fields used by SRP05.
7,No duplicate episode/domain records,True,0,360,Duplicate records can otherwise overweight par...
8,Expected task count,True,0,3,"Observed tasks: ['lorenz_x', 'mackey_glass', '..."
9,Expected reservoir-seed count,True,0,3,"Observed seeds: [np.int64(0), np.int64(1), np...."


In [12]:

# ============================================================
# Result-to-source map and exact manuscript statements
# ============================================================
source_rows = [
    {
        "derived_output": "srp05_task_table.csv",
        "source_file": task_source_name,
        "source_columns": (
            "task, domain, historical_nrmse, robust_sparse_nrmse, "
            "meta_top3_nrmse, oracle_sparse_nrmse"
        ),
        "derivation": (
            "Global-domain exact task means and relative gains: "
            "(historical - policy) / historical."
        ),
    },
    {
        "derived_output": "srp05_certification_cost_table.csv",
        "source_file": task_source_name,
        "source_columns": "task, domain, oracle_sparse_nrmse",
        "derivation": (
            "(certified oracle - global oracle) / global oracle; "
            "checked against safety summary when available."
        ),
    },
]

if safety_source_name:
    source_rows.append({
        "derived_output": "srp05_certification_cost_table.csv",
        "source_file": safety_source_name,
        "source_columns": "task, relative_price_of_safety",
        "derivation": "Independent source-value comparison.",
    })

if episode_source_name:
    source_rows.extend([
        {
            "derived_output": "srp05_shift_family_table.csv",
            "source_file": episode_source_name,
            "source_columns": (
                "task, seed, family, domain, historical_nrmse, "
                "robust_sparse_nrmse"
            ),
            "derivation": (
                "Global-domain family aggregation; bootstrap over task–seed clusters; "
                "episode and cluster harm fractions."
            ),
        },
        {
            "derived_output": "srp05_task_seed_metrics.csv",
            "source_file": episode_source_name,
            "source_columns": (
                "task, seed, domain and policy NRMSE columns"
            ),
            "derivation": "Mean within task × seed × action-space domain.",
        },
        {
            "derived_output": "srp05_validity_checks.csv",
            "source_file": episode_source_name,
            "source_columns": (
                "domain, historical_nrmse, oracle_sparse_nrmse"
            ),
            "derivation": "Matched-action-space inequality checks per held-out record.",
        },
    ])

source_map = pd.DataFrame(source_rows)
source_map.to_csv(
    OUTPUT_DIR / "srp05_result_source_map.csv",
    index=False,
)

lines: List[str] = []
lines.append("SRP05 MANUSCRIPT-INTEGRATION REPORT")
lines.append("=" * 44)
lines.append("")
lines.append(
    "All relative gains use (L_historical - L_policy) / L_historical. "
    "Positive values indicate lower NRMSE than the historically best fixed candidate."
)
lines.append(
    "The target-fitted sparse oracle is retrospective and uses held-out labels. "
    "It is not a deployable test-time policy."
)
lines.append(
    "The multidimensional historically screened subset is denoted Theta_cert, "
    "not T_safe."
)
lines.append("")

lines.append("TASK-LEVEL RESULTS")
lines.append("-" * 18)
for row in task_table.itertuples(index=False):
    lines.append(
        f"{row.task}: historical fixed {row.historical_best_fixed_nrmse:.6f}; "
        f"historically fitted sparse {row.historically_fitted_sparse_nrmse:.6f} "
        f"({100.0 * row.historical_sparse_relative_gain:.2f}% relative gain); "
        f"target-fitted sparse oracle {row.target_fitted_sparse_oracle_nrmse:.6f} "
        f"({100.0 * row.target_oracle_relative_gain:.2f}% retrospective opportunity)."
    )
    if np.isfinite(row.label_free_meta_selector_nrmse):
        lines.append(
            f"  Label-free meta-risk selector: "
            f"{row.label_free_meta_selector_nrmse:.6f} "
            f"({100.0 * row.meta_selector_relative_gain:.2f}% relative gain)."
        )
lines.append("")

if not shift_family_table.empty:
    lines.append("SHIFT-FAMILY RESULTS")
    lines.append("-" * 20)
    for row in shift_family_table.itertuples(index=False):
        lines.append(
            f"{row.shift_family}: historically fitted sparse aggregation changed "
            f"NRMSE by {100.0 * row.historical_sparse_relative_gain:.2f}% relative "
            f"to historical fixed "
            f"(cluster bootstrap {100.0 * row.historical_sparse_relative_gain_ci_low:.2f}% "
            f"to {100.0 * row.historical_sparse_relative_gain_ci_high:.2f}%); "
            f"it was worse on {row.episode_harm_count}/{row.episode_window_count} "
            f"held-out windows ({100.0 * row.episode_harm_fraction:.1f}%)."
        )
    lines.append("")

if not certification_table.empty:
    lines.append("PRICE OF CERTIFICATION")
    lines.append("-" * 22)
    for row in certification_table.itertuples(index=False):
        lines.append(
            f"{row.task}: restricting the retrospective sparse oracle from "
            f"Theta_global to Theta_cert increased mean NRMSE by "
            f"{100.0 * row.relative_certification_cost_recomputed:.2f}%."
        )
    lines.append("")

lines.append("CAUTIOUS MANUSCRIPT INTERPRETATION")
lines.append("-" * 34)
lines.append(
    "The audit establishes substantial target-specific opportunity within the "
    "two-dimensional temperature–leak structural bank, but it does not establish "
    "that the best target-specific mixture can be selected without labels."
)
lines.append(
    "A sparse aggregation fitted only on historical shifted environments recovered "
    "part of the retrospective opportunity on average, while its benefit depended "
    "on shift family and could reverse under some conditions."
)
lines.append(
    "The label-free meta-risk selector did not consistently improve on the simpler "
    "historically fitted sparse aggregation, supporting a boundary claim rather "
    "than a new autonomous adaptation method."
)
lines.append("")
lines.append("VALIDITY STATUS")
lines.append("-" * 15)

for row in validity_checks.itertuples(index=False):
    if pd.isna(row.passed):
        status = "N/A"
    else:
        status = "PASS" if bool(row.passed) else "FAIL"
    lines.append(f"{status}: {row.check} — {row.detail}")

report_text = "\n".join(lines) + "\n"
(OUTPUT_DIR / "srp05_manuscript_statements.txt").write_text(
    report_text,
    encoding="utf-8",
)

print(report_text)


SRP05 MANUSCRIPT-INTEGRATION REPORT

All relative gains use (L_historical - L_policy) / L_historical. Positive values indicate lower NRMSE than the historically best fixed candidate.
The target-fitted sparse oracle is retrospective and uses held-out labels. It is not a deployable test-time policy.
The multidimensional historically screened subset is denoted Theta_cert, not T_safe.

TASK-LEVEL RESULTS
------------------
lorenz_x: historical fixed 0.657726; historically fitted sparse 0.527367 (19.82% relative gain); target-fitted sparse oracle 0.436730 (33.60% retrospective opportunity).
  Label-free meta-risk selector: 0.542231 (17.56% relative gain).
mackey_glass: historical fixed 0.619260; historically fitted sparse 0.495867 (19.93% relative gain); target-fitted sparse oracle 0.377551 (39.03% retrospective opportunity).
  Label-free meta-risk selector: 0.540338 (12.74% relative gain).
narma10: historical fixed 1.006426; historically fitted sparse 0.885420 (12.02% relative gain); targe

In [13]:

# ============================================================
# Reproducibility manifest and ZIP bundle
# ============================================================
manifest = {
    "audit": "SRP05_MANUSCRIPT_INTEGRATION_AUDIT",
    "input_directory": str(INPUT_DIR.resolve()),
    "task_summary_source": task_source_name,
    "episode_policy_source": episode_source_name,
    "safety_summary_source": safety_source_name,
    "bootstrap_replicates": N_BOOT,
    "bootstrap_unit": (
        "task-seed clusters for shift-family summaries; "
        "seed clusters for task-specific descriptive intervals"
    ),
    "confidence_level": CI_LEVEL,
    "numeric_tolerance": NUMERIC_TOL,
    "domain_renaming": {
        "global": "Theta_global",
        "safe/certified": "Theta_cert",
    },
    "formulas": {
        "relative_gain": "(L_historical - L_policy) / L_historical",
        "relative_certification_cost": (
            "(L_oracle_certified - L_oracle_global) / L_oracle_global"
        ),
    },
    "summary_only_mode": episode_policy is None,
}

(OUTPUT_DIR / "srp05_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

zip_path = OUTPUT_DIR.with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted(OUTPUT_DIR.rglob("*")):
        if file_path.is_file():
            archive.write(
                file_path,
                arcname=str(file_path.relative_to(OUTPUT_DIR)),
            )

print("Saved output directory:", OUTPUT_DIR.resolve())
print("Saved ZIP bundle:", zip_path.resolve())
print("\nPlease upload the executed notebook and the ZIP bundle.")


Saved output directory: /home/ildefons/agentecon/SRP05_outputs
Saved ZIP bundle: /home/ildefons/agentecon/SRP05_outputs.zip

Please upload the executed notebook and the ZIP bundle.



## Interpretation checklist

Before transferring any number to the paper:

1. All matched-action-space oracle checks should pass.
2. `srp05_task_table.csv` is the primary task-level source.
3. `srp05_shift_family_table.csv` is used only when an episode-level policy table was found.
4. With only three reservoir seeds, task-specific bootstrap intervals are descriptive; do not present them as strong asymptotic evidence.
5. The target-fitted sparse oracle is an upper bound on opportunity, not an operational method.
6. Use \(\Theta_{\mathrm{cert}}\) for the multidimensional historically screened set, preserving \(\mathcal{T}_{\mathrm{safe}}\) for the original one-dimensional temperature safe region.
7. A negative shift-family gain is a substantive limitation and must not be averaged away.
